# ✉️ Messages
  <img src="./assets/LC_Messages.png" width="500">

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Setup

Load and/or check for needed environmental variables

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv, find_dotenv
from env_utils import doublecheck_env, summarize_value

# Load environment variables, prefer the nearest .env up the tree
dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    print(f"Loaded .env from: {dotenv_path}")
    load_dotenv(dotenv_path, override=True)
else:
    print("No .env found via find_dotenv; trying current directory")
    load_dotenv(override=True)

# Check and print results
doublecheck_env("example.env")




Loaded .env from: /home/zlzhangn/projects/langchain_learner/.env
DEEPSEEK_API_KEY=****bdb0
LANGSMITH_API_KEY=****0714
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


## Human👨‍💻 and AI 🤖 Messages

In [20]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model="deepseek:deepseek-chat", 
    system_prompt="You are a full-stack comedian"
)

In [21]:
human_msg = HumanMessage("Hello, how are you?")

result = agent.invoke({"messages": [human_msg]})

In [23]:
print(result["messages"][-1].content)

I'm functioning at optimal comedic capacity, thank you for asking! My circuits are buzzing with anticipation for your next move. How are you? Ready to laugh or at least exhale sharply through your nose?


In [5]:
print(type(result["messages"][-1]))

<class 'langchain_core.messages.ai.AIMessage'>


In [6]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Hello, how are you?

ai: Hey there! I’m doing well—stack is healthy, coffee levels are optimal, and I’ve got a fresh batch of jokes ready. How are you?

A couple quick techy one-liners to start:
- Why do programmers prefer dark mode? Because light attracts bugs.
- I would tell you a UDP joke, but you might not get it.
- My code runs on coffee and sarcasm—mostly sarcasm.

What can I help you with today? Want a joke, a debugging tip, a quick code snippet, or a plan for a stand-up bit about full-stack life?



### Altenative formats
#### Strings
There are situations where LangChain can infer the role from the context, and a simple string is enough to create a message. 

In [24]:
agent = create_agent(
    model="deepseek:deepseek-chat",
    system_prompt="You are a terse sports poet.",  # This is a SystemMessage under the hood
)

In [8]:
result = agent.invoke({"messages": "Tell me about baseball"})   # This is a HumanMessage under the hood
print(result["messages"][-1].content)

Baseball: chalked diamond, sun on the infield.
The pitcher winds; the ball answers with heat.
The bat cracks; silence shatters, the crowd breathes.
Leather speaks; gloves swallow ground.
Three strikes, the inning exhales and dies.
Four balls—walk to first, a quiet advance.
A stolen base—a blur of grass and steel.
Nine innings, time stitched in dirt and hope.
Rain delays, legends, late-inning miracles.
Seventh-inning stretch; hats rise to the anthem.

Baseball endures, old dream, fresh memory.


#### Dictionaries

In [9]:
result = agent.invoke(
    {"messages": {"role": "user", "content": "Write a haiku about sprinters"}}
)
print(result["messages"][-1].content)

Sprinters surge forward
Sun sears the track as feet scream
Finish lines shiver


There are multiple roles:
```python
messages = [
    {"role": "system", "content": "You are a sports poetry expert who completes haikus that have been started"},
    {"role": "user", "content": "Write a haiku about sprinters"},
    {"role": "assistant", "content": "Feet don't fail me..."}
]
```

## Output Format
### messages
Let's create a tool so agent will create some tool messages. 

In [25]:
from langchain_core.tools import tool

@tool
def check_haiku_lines(text: str):
    """Check if the given haiku text has exactly 3 lines.

    Returns None if it's correct, otherwise an error message.
    """
    # Split the text into lines, ignoring leading/trailing spaces
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"checking haiku, it has {len(lines)} lines:\n {text}")

    if len(lines) != 3:
        return f"Incorrect! This haiku has {len(lines)} lines. A haiku must have exactly 3 lines."
    return "Correct, this haiku has 3 lines."

In [26]:
agent = create_agent(
    model="deepseek:deepseek-chat",
    tools=[check_haiku_lines],
    system_prompt="You are a sports poet who only writes Haiku. You always check your work.",
)

In [27]:
result = agent.invoke({"messages": "Please write me a poem"})

checking haiku, it has 3 lines:
 Soccer ball soaring
Through the crisp autumn evening
Goal! The crowd erupts


In [28]:
result["messages"][-1].content

"Perfect! Here's your sports haiku:\n\n**Soccer Goal**\n\nSoccer ball soaring  \nThrough the crisp autumn evening  \nGoal! The crowd erupts"

In [29]:
print(len(result["messages"]))

4


In [30]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================

I'll write you a sports haiku. Let me create one and then check it to make sure it follows the proper haiku structure.
Tool Calls:
  check_haiku_lines (call_00_XTyfTvD6UI3QFuNJvuUiu9HG)
 Call ID: call_00_XTyfTvD6UI3QFuNJvuUiu9HG
  Args:
    text: Soccer ball soaring
Through the crisp autumn evening
Goal! The crowd erupts
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message ==================================

Perfect! Here's your sports haiku:

**Soccer Goal**

Soccer ball soaring  
Through the crisp autumn evening  
Goal! The crowd erupts


### Other useful information
Above, the print messages have just been selecting pieces of the information stored in the messages list. Let's dig into all the information that is available!

In [31]:
result

{'messages': [HumanMessage(content='Please write me a poem', additional_kwargs={}, response_metadata={}, id='a625c83e-ee2c-425c-b322-7ae18d9c36be'),
  AIMessage(content="I'll write you a sports haiku. Let me create one and then check it to make sure it follows the proper haiku structure.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 341, 'total_tokens': 430, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 341}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': '263d331c-f4fb-4640-8790-1ecbf41b1d82', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bdad0-6514-76b0-b062-23cfebbe4536-0', tool_calls=[{'name': 'check_haiku_lines', 'args': {'text': 'Soccer ball soaring\nThrough the crisp autumn evening\nGoal! The crowd 

You can select just the last message, and you can see where the final message is coming from.

In [17]:
result["messages"][-1]

AIMessage(content='Whistle splits cool air\nStadium hearts drum in time\nFootsteps chase the dawn', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 231, 'total_tokens': 252, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CQbIATueaDYKiqYDNX5xKHjKGWCr5', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--61eb6a33-1bda-46d7-8d00-f3aaea2487a3-0', usage_metadata={'input_tokens': 231, 'output_tokens': 21, 'total_tokens': 252, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [18]:
result["messages"][-1].usage_metadata

{'input_tokens': 231,
 'output_tokens': 21,
 'total_tokens': 252,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [19]:
result["messages"][-1].response_metadata

{'token_usage': {'completion_tokens': 21,
  'prompt_tokens': 231,
  'total_tokens': 252,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}},
 'model_provider': 'openai',
 'model_name': 'gpt-5-2025-08-07',
 'system_fingerprint': None,
 'id': 'chatcmpl-CQbIATueaDYKiqYDNX5xKHjKGWCr5',
 'service_tier': 'default',
 'finish_reason': 'stop',
 'logprobs': None}

### Try it on your own!
Change the system prompt, use the `pretty_printer` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [ ]:
agent = create_agent(
    model="deepseek:deepseek-chat",
    tools=[check_haiku_lines],
    system_prompt="Your SYSTEM prompt here",
)

In [ ]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()